# BT4012 Kaggle Competition 2026: Fraudulent Bitcoin Transaction Detection

**Final model (top submission, public ROC-AUC 0.9687):**
LightGBM (5-seed average) on raw + graph-engineered features, trained with SMOTE oversampling (illicit : licit = 1 : 1).

| Section | Content |
|---|---|
| 1 | Setup and data loading |
| 2 | Data checks and preprocessing |
| 3 | Graph preparation (consistent "known-node" neighbourhood) |
| 4 | Feature engineering: structure, neighbour feature averages, neighbour predictions (leakage-safe) |
| 5 | Model: LightGBM seed ensemble + SMOTE |
| 6 | Validation: expanding time windows |
| 7 | Final training and submission file |

**Reproducibility:** fixed random seeds throughout. Expected folder layout:
```
repo/
├── e1234567.ipynb
├── requirements.txt
├── data/          train.csv, test.csv, txs_edgelist.csv, sample_submission.csv  (from Kaggle, not committed)
└── submissions/   created by this notebook
```

## 1. Setup and data loading

In [17]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE

SEED = 42
DATA_DIR = "data" if os.path.isdir("data") else "../data"     # works from repo root or a sub-folder
SUB_DIR = "submissions"
os.makedirs(SUB_DIR, exist_ok=True)

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feat_cols = [f"feat_{i}" for i in range(1, 166)]               # 165 anonymised, pre-normalised features
print(f"train {train.shape} | test {test.shape} | edges {edges.shape}")

train (141772, 168) | test (15329, 168) | edges (234355, 2)


## 2. Data checks and preprocessing

- **Labels:** `label` is `1` (illicit), `0` (licit) or the text `none` (unknown). `none` is converted to `NaN`by converting `label` to numeric. 
  Only labelled rows are used for supervised training (pseudo-labelling the unknown rows was tested and did not improve the performance).
- **Missing values:** none in the features, so no imputation is needed.
- **Scaling:** not needed for this dataset. Features are already normalised and tree models are scale-invariant.
- **Column `time_step`** is **not** used as a feature: test steps (36–49) never occur in training.
  It is used only for time-based validation and for leakage control.

In [18]:
train["label"] = pd.to_numeric(train["label"], errors="coerce")       # "none" -> NaN

lab = train[train["label"].notna()].copy()
lab["label"] = lab["label"].astype(int)

print("Label counts (NaN = unknown):", train["label"].value_counts(dropna=False).to_dict())
print(f"Illicit rate among labelled: {lab['label'].mean():.2%}")
print("Missing feature values  train:", int(train[feat_cols].isna().sum().sum()),
      "| test:", int(test[feat_cols].isna().sum().sum()))
print("Train time steps:", train["time_step"].min(), "-", train["time_step"].max(),
      "| Test time steps:", test["time_step"].min(), "-", test["time_step"].max())

Label counts (NaN = unknown): {nan: 110537, 0.0: 27591, 1.0: 3644}
Illicit rate among labelled: 11.67%
Missing feature values  train: 0 | test: 0
Train time steps: 1 - 35 | Test time steps: 36 - 49


**Homophily check (motivation for the graph features).** 

**Dyadic Test:** Does the whole network have fewer fraud–legit edges than random labelling would give?

In [7]:
lab_map = lab.set_index("txId")["label"]            # labelled training transactions only

# Node shares (as on the slide)
n_fraud = int((lab_map == 1).sum())
n_legit = int((lab_map == 0).sum())
n = n_fraud + n_legit
x, y = n_legit / n, n_fraud / n
expected = 2 * x * y                                # cross-edge share if labels were random

# Edges with both ends labelled
e_lab = edges.assign(a=edges["txId1"].map(lab_map), b=edges["txId2"].map(lab_map)).dropna(subset=["a", "b"])
m = len(e_lab)
cross      = int((e_lab["a"] != e_lab["b"]).sum())
same_fraud = int(((e_lab["a"] == 1) & (e_lab["b"] == 1)).sum())
same_legit = int(((e_lab["a"] == 0) & (e_lab["b"] == 0)).sum())
r = cross / m

print(f"Nodes: {n_legit:,} legitimate, {n_fraud:,} fraudulent  ->  x = {x:.3f}, y = {y:.3f}")
print(f"2xy (expected cross-edge share) = {expected:.3f}")
print(f"Edges (both ends labelled): {m:,}")
print(f"  cross-labelled: {cross:,} | same-labelled fraud: {same_fraud:,} | same-labelled legit: {same_legit:,}")
print(f"r (observed cross-edge share) = {r:.3f}")
print("=> homophilic" if r < expected else "=> not homophilic")

#is the gap statistically significant? (z-test against the random-label expectation)
z = (cross - m * expected) / np.sqrt(m * expected * (1 - expected))
print(f"z = {z:.1f}  (z < -1.96 means significantly fewer cross edges than chance, at the 5% level)")

Nodes: 27,591 legitimate, 3,644 fraudulent  ->  x = 0.883, y = 0.117
2xy (expected cross-edge share) = 0.206
Edges (both ends labelled): 23,900
  cross-labelled: 1,271 | same-labelled fraud: 857 | same-labelled legit: 21,772
r (observed cross-edge share) = 0.053
=> homophilic
z = -58.4  (z < -1.96 means significantly fewer cross edges than chance, at the 5% level)


Among edges where both ends are labelled,
an illicit transaction's neighbour is illicit far more often than the base rate. A licit transaction's neighbour rarely is.

In [8]:
lab_map = lab.set_index("txId")["label"]
e_lab = edges.assign(a=edges["txId1"].map(lab_map), b=edges["txId2"].map(lab_map)).dropna(subset=["a", "b"])
print(f"Labelled-labelled edges: {len(e_lab):,}   (base illicit rate {lab['label'].mean():.1%})")
for a, b, name in [("a", "b", "source -> target"), ("b", "a", "target <- source")]:
    print(f"{name}: neighbour illicit | node illicit = {e_lab.loc[e_lab[a] == 1, b].mean():.1%}, "
          f"| node licit = {e_lab.loc[e_lab[a] == 0, b].mean():.1%}")

Labelled-labelled edges: 23,900   (base illicit rate 11.7%)
source -> target: neighbour illicit | node illicit = 56.0%, | node licit = 2.7%
target <- source: neighbour illicit | node illicit = 58.9%, | node licit = 3.0%


- **Downstream:** if a transaction is illicit, the transactions that spend its money are illicit 56.0% of the time. For a licit transaction, only 2.7%.
- **Upstream:** if a transaction is illicit, the transactions that funded it are illicit 58.9% of the time. For a licit transaction, only 3.0%.

## 3. Graph preparation

### Why neighbour features use only "known" transactions

In the data, every time step contains two kinds of transactions: **labelled** ones (illicit/licit,
decided by the dataset's labelling heuristics) and **unknown** ones (class undetermined, even by the organisers).

- **Training period (steps 1–35):** both kinds are provided in `train.csv` (unknown = `label == "none"`).
- **Test period (steps 36–49):** I made the assumption that only the labelled transactions are provided, in `test.csv` (labels hidden for scoring).
  There are 46,668 unknown test-period transactions that appear in  `txs_edgelist.csv`, but not `test.csv`, hence have no features.

Edges only connect transactions **within the same time step**, so there are **no edges between the training and test
periods**. The edge list breaks down as:

| Edge type | Count |
|---|---|
| Both ends in `train.csv` | 163,194 |
| Both ends in `test.csv` | 12,724 |
| Between a train and a test transaction | 0 |
| Touching a removed (unknown, test-period) transaction | 58,437 |

As a result, a **test** transaction can only "see" neighbours that are other **test** transactions; its unknown neighbours
have no features. Hence, to keep features **consistent** between the two periods, every neighbour-based feature uses only
**known transactions**:

- for a **training** transaction: its **labelled** training neighbours (its unknown neighbours are ignored),
- for a **test** transaction: its **test** neighbours (its unknown neighbours are unavailable anyway).

Both periods then follow the same rule: *neighbours whose features exist and whose class is labelled (or hidden-labelled)*.

**Exception: degrees.** The edge list still contains every edge, including those to removed transactions, so in-degree
and out-degree are computed from all edges and are complete in both periods.

In [9]:
nodes = pd.concat([lab, test.drop(columns="index")], ignore_index=True).set_index("txId").copy()
nodes["is_test"] = nodes.index.isin(test["txId"])

# Known-node neighbour pairs, both directions ("in": nbr pays into node, "out": node pays into nbr)
e = edges[edges["txId1"].isin(nodes.index) & edges["txId2"].isin(nodes.index)]
nbr = pd.concat([
    pd.DataFrame({"node": e["txId2"].values, "nbr": e["txId1"].values, "dir": "in"}),
    pd.DataFrame({"node": e["txId1"].values, "nbr": e["txId2"].values, "dir": "out"}),
], ignore_index=True)
print(f"Known nodes: {len(nodes):,} | known-node edges: {len(e):,}")

Known nodes: 46,564 | known-node edges: 36,624


## 4. Feature engineering

### 4a. Structural features (5)

In [10]:
nodes["in_deg"]  = edges.groupby("txId2").size().reindex(nodes.index).fillna(0)   # transactions feeding into this one
nodes["out_deg"] = edges.groupby("txId1").size().reindex(nodes.index).fillna(0)   # transactions spending its outputs
nodes["deg"]     = nodes["in_deg"] + nodes["out_deg"]
nodes["n_known_nbrs"]    = nbr.groupby("node").size().reindex(nodes.index).fillna(0)
nodes["frac_known_nbrs"] = nodes["n_known_nbrs"] / nodes["deg"].clip(lower=1)
STRUCT_COLS = ["in_deg", "out_deg", "deg", "n_known_nbrs", "frac_known_nbrs"]

# Coverage is similar in train and test, so these features mean the same thing in both periods
print(nodes.groupby("is_test")[["deg", "n_known_nbrs", "frac_known_nbrs"]].mean().round(3))

           deg  n_known_nbrs  frac_known_nbrs
is_test                                      
False    2.866          1.53            0.631
True     3.242          1.66            0.609


### 4b. Neighbour feature averages (80)

For the **40 most important raw features**, the mean and max over each transaction's known neighbours.
Importance comes from a LightGBM model trained on **steps 1–15 only**: every validation fold starts at
step 16 or later, so this choice never sees validation labels. Only feature values are used (never labels),
so these can be computed once for all nodes.

In [11]:
def lgbm(seed=SEED):
    """Base LightGBM configuration used throughout."""
    return lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=31,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
                              random_state=seed, verbose=-1)

early = nodes[~nodes["is_test"] & (nodes["time_step"] <= 15)]
imp = pd.Series(lgbm().fit(early[feat_cols], early["label"]).feature_importances_,
                index=feat_cols).sort_values(ascending=False)
TOP40 = imp.head(40).index.tolist()

vals = nodes.loc[nbr["nbr"], TOP40].set_index(nbr["node"].values)
agg = vals.groupby(level=0).agg(["mean", "max"])
agg.columns = [f"nbr40_{stat}_{c}" for c, stat in agg.columns]
NBR40_COLS = list(agg.columns)
nodes = nodes.join(agg)
nodes[NBR40_COLS] = nodes[NBR40_COLS].fillna(-999)       # -999 = no known neighbour

L = nodes[~nodes["is_test"]].copy()                      # labelled training transactions
T = nodes[nodes["is_test"]].copy()                       # test transactions
print("Top-5 features by importance (steps 1-15):", TOP40[:5])
print("L:", L.shape, "| T:", T.shape)

Top-5 features by importance (steps 1-15): ['feat_2', 'feat_58', 'feat_52', 'feat_53', 'feat_142']
L: (31235, 253) | T: (15329, 253)


### 4c. Neighbour-prediction features (9), two-stage and leakage-safe

- **Stage 1:** a LightGBM model on raw features predicts a fraud probability for every transaction.
- **Stage 2 features:** summaries of the neighbours' predicted probabilities:

| Feature | Meaning |
|---|---|
| `nbr_p_max` | Score of the **most suspicious** neighbour (either direction) |
| `nbr_p_mean` | **Average** neighbour score |
| `nbr_p_sum` | **Total** suspicion around the transaction (grows with more suspicious neighbours) |
| `nbr_n_high` | **How many** neighbours score above 0.5 |
| `nbr_p_in_max` | Most suspicious transaction that **sent money to it** (upstream) |
| `nbr_p_out_max` | Most suspicious transaction that **received money from it** (downstream) |
| `nbr2_p_max`, `nbr2_p_mean` | Highest and average score among **neighbours of neighbours** (2 hops away) |
| `nbr2_n` | **How many** 2-hop neighbours there are |

*Scores are stage-1 fraud probabilities. A value of −1 means the transaction has no known neighbour of that kind.*

**Leakage control:** a training transaction's own label must not reach its neighbours' scores.
Neighbours always share a time step, so stage-1 scores for training rows are **out-of-fold, grouped by time step**:
each step is scored by a model that never saw that step. Rows being scored (validation or test) are predicted by a
stage-1 model trained on all training rows.

In [12]:
def stage1_preds(fit, apply):
    """Fraud probability for every node in fit ∪ apply (training rows out-of-fold by time step)."""
    p_fit = pd.Series(np.nan, index=fit.index)
    for tr_idx, oof_idx in GroupKFold(n_splits=5).split(fit, groups=fit["time_step"]):
        m = lgbm().fit(fit.iloc[tr_idx][feat_cols], fit.iloc[tr_idx]["label"])
        p_fit.iloc[oof_idx] = m.predict_proba(fit.iloc[oof_idx][feat_cols])[:, 1]
    m = lgbm().fit(fit[feat_cols], fit["label"])
    p_apply = pd.Series(m.predict_proba(apply[feat_cols])[:, 1], index=apply.index)
    return pd.concat([p_fit, p_apply])

def nbr_pred_features(fit, apply):
    """1-hop and 2-hop aggregates of neighbours' stage-1 fraud probabilities."""
    p = stage1_preds(fit, apply)
    nb = nbr[nbr["node"].isin(p.index) & nbr["nbr"].isin(p.index)].copy()
    nb["p"] = p.reindex(nb["nbr"]).values
    g = nb.groupby("node")["p"]
    out = pd.DataFrame({"nbr_p_max": g.max(), "nbr_p_mean": g.mean(), "nbr_p_sum": g.sum(),
                        "nbr_n_high": nb.assign(h=nb["p"] > 0.5).groupby("node")["h"].sum()})
    out["nbr_p_in_max"]  = nb[nb["dir"] == "in"].groupby("node")["p"].max()     # where money came from
    out["nbr_p_out_max"] = nb[nb["dir"] == "out"].groupby("node")["p"].max()    # where money went
    pairs = nb[["node", "nbr"]]                                                  # 2-hop: neighbours of neighbours
    two = pairs.merge(pairs.rename(columns={"node": "nbr", "nbr": "nbr2"}), on="nbr")
    two = two[two["nbr2"] != two["node"]].drop_duplicates(["node", "nbr2"])
    two["p"] = p.reindex(two["nbr2"]).values
    g2 = two.groupby("node")["p"]
    out["nbr2_p_max"], out["nbr2_p_mean"], out["nbr2_n"] = g2.max(), g2.mean(), g2.size()
    return out.reindex(p.index).fillna(-1)                                       # -1 = no such neighbour

def build_features(fit, apply):
    """Full feature matrix: raw (165) + structure (5) + neighbour averages (80) + neighbour predictions (9)."""
    base = feat_cols + STRUCT_COLS + NBR40_COLS
    f = nbr_pred_features(fit, apply)
    return fit[base].join(f.loc[fit.index]), apply[base].join(f.loc[apply.index])

def raw_features(fit, apply):
    return fit[feat_cols], apply[feat_cols]

## 5. Model: LightGBM seed ensemble + SMOTE

- **Seed averaging:** LightGBM trained with several random seeds, predictions averaged. This reduces run-to-run noise.
- **SMOTE:** synthetic illicit examples are generated on the **training rows only** until illicit : licit = 1 : 1.
  Validation and test rows are never resampled. On validation, SMOTE raised the graph model's AUC from 0.949 to
  0.954–0.957 (the range depends on the random draw); class weighting (`scale_pos_weight`) did not help.

In [13]:
class SeedAvg:
    """LightGBM averaged over several random seeds."""
    def __init__(self, seeds=(0, 1, 2)):
        self.seeds = seeds
    def fit(self, X, y):
        self.models = [lgbm(seed=s).fit(X, y) for s in self.seeds]
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

class SMOTEModel:
    """Oversample the illicit class with SMOTE (training data only), then fit the wrapped model."""
    def __init__(self, make_model, ratio=1.0):
        self.make_model, self.ratio = make_model, ratio
    def fit(self, X, y):
        X_res, y_res = SMOTE(sampling_strategy=self.ratio, k_neighbors=5, random_state=SEED).fit_resample(X, y)
        self.model = self.make_model().fit(X_res, y_res)
        return self
    def predict_proba(self, X):
        return self.model.predict_proba(X)

## 6. Validation: expanding time windows

The test set lies **after** the training period, so validation mimics that: train on all steps up to a cutoff,
validate on the following ~10 steps (the real test horizon is 14 steps). No random or stratified splits, which would
leak future transactions into training. Each validation block contains several hundred illicit transactions.

All features that depend on labels (the neighbour predictions) are rebuilt **inside each fold** from that
fold's training rows only. Models use 3 seeds here; the final model uses 5.

*Runtime: roughly 10–20 minutes for this section. It can be skipped to go straight to Section 7.*

In [14]:
FOLDS = [(15, range(16, 26)), (20, range(21, 31)), (25, range(26, 36))]   # (last training step, validation steps)

for last_train, val_steps in FOLDS:
    tr, va = L[L["time_step"] <= last_train], L[L["time_step"].isin(val_steps)]
    print(f"train 1-{last_train}: {len(tr):,} rows ({tr['label'].mean():.1%} illicit) -> "
          f"val {val_steps.start}-{val_steps.stop - 1}: {len(va):,} rows, {int(va['label'].sum())} illicit")

def evaluate(name, make_model, build):
    aucs = []
    for last_train, val_steps in FOLDS:
        fit = L[L["time_step"] <= last_train]
        va  = L[L["time_step"].isin(val_steps)]
        X_fit, X_va = build(fit, va)
        model = make_model().fit(X_fit, fit["label"])
        aucs.append(roc_auc_score(va["label"], model.predict_proba(X_va)[:, 1]))
    print(f"{name:42s} folds={np.round(aucs, 4)}  mean={np.mean(aucs):.4f}")
    return {"model": name, **{f"fold{i + 1}": round(a, 4) for i, a in enumerate(aucs)}, "mean_AUC": round(np.mean(aucs), 4)}

results = [
    evaluate("Baseline: raw features",          SeedAvg,                           raw_features),
    evaluate("Graph features",                  SeedAvg,                           build_features),
    evaluate("Graph features + SMOTE (final)",  lambda: SMOTEModel(SeedAvg, 1.0),  build_features),
]
pd.DataFrame(results)

train 1-15: 15,514 rows (7.4% illicit) -> val 16-25: 8,686 rows, 1185 illicit
train 1-20: 18,889 rows (9.4% illicit) -> val 21-30: 8,016 rows, 1183 illicit
train 1-25: 24,200 rows (9.7% illicit) -> val 26-35: 7,035 rows, 1307 illicit
Baseline: raw features                     folds=[0.9598 0.9054 0.9969]  mean=0.9540
Graph features                             folds=[0.9575 0.8927 0.9981]  mean=0.9494
Graph features + SMOTE (final)             folds=[0.9663 0.9072 0.9982]  mean=0.9572


,model,fold1,fold2,fold3,mean_AUC
0,Baseline: raw features,0.9598,0.9054,0.9969,0.9540
1,Graph features,0.9575,0.8927,0.9981,0.9494
2,Graph features + SMOTE (final),0.9663,0.9072,0.9982,0.9572


**Public leaderboard (30% of test) for the same progression**, from the experiment log:

| Submission | Public ROC-AUC |
|---|---|
| Baseline LightGBM, raw features | 0.9470 |
| + neighbour predictions (1-hop) | 0.9534 |
| + 2-hop neighbour predictions | 0.9549 |
| + neighbour feature averages (top-40) | 0.9613 |
| **+ SMOTE 1:1 (final)** | **0.9687** |

Validation did not always rank ideas the same way as the leaderboard , so both were used when choosing the final model.

## 7. Final training and submission

Features are rebuilt with **all** labelled rows (steps 1–35) as the training set and the test rows as the apply set.
The final model is SMOTE (1 : 1) + LightGBM averaged over 5 seeds.

In [15]:
X_fit, X_test = build_features(L, T)
final_model = SMOTEModel(lambda: SeedAvg(seeds=(0, 1, 2, 3, 4)), ratio=1.0).fit(X_fit, L["label"])

pred = final_model.predict_proba(X_test.loc[test["txId"]])[:, 1]
submission = pd.DataFrame({"index": test["index"].values, "target": pred})

# Format checks against the sample submission
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample) and set(submission["index"]) == set(sample["index"])
assert submission["target"].between(0, 1).all() and submission["target"].notna().all()

out_path = f"{SUB_DIR}/final_submission.csv"
submission.to_csv(out_path, index=False)
print(f"Saved {out_path}  ({len(submission):,} rows, {X_fit.shape[1]} features)")
submission.head()

Saved submissions/final_submission.csv  (15,329 rows, 259 features)


,index,target
0,0,0.000046
1,1,0.000584
2,2,0.000494
3,3,0.000120
4,4,0.000102


Optional sanity check: if the earlier Kaggle file `26_lgbm5_graph40_smote.csv` is present, the rankings should match.

In [16]:
from scipy.stats import spearmanr

ref_path = f"{SUB_DIR}/26_lgbm5_graph40_smote.csv"
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path).set_index("index")["target"]
    corr = spearmanr(submission.set_index("index")["target"], ref.loc[submission["index"]]).correlation
    print(f"Rank correlation with submission 26: {corr:.4f}")
else:
    print("Reference file not found - skipping.")

Rank correlation with submission 26: 1.0000
